In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import Conv2D,InputLayer,Dense,MaxPooling2D

In [ ]:
dataset, dataset_info = tfds.load('malaria', 
                                as_supervised=True, 
                                with_info=True, 
                                shuffle_files=True, 
                                split='train')

In [ ]:
for image, label in dataset.take(1): 
    print(image, label)


In [ ]:
class_names = dataset_info.features['label'].names

In [ ]:
class_names

In [ ]:
def splits(dataset,train_ratio,val_ratio,test_ratio):
  dataset_size=len(dataset)
  train_dataset=dataset.take(int(train_ratio*dataset_size))
  val_test_dataset=dataset.skip(int(train_ratio*dataset_size))
  val_dataset=val_test_dataset.take(int(val_ratio*dataset_size))
  test_dataset=val_test_dataset.skip(int(val_ratio*dataset_size))
  return train_dataset,val_dataset,test_dataset

In [ ]:
train_dataset,val_dataset,test_dataset=splits(dataset,0.6,0,0.2)

# Data Visualization

In [ ]:
for i,(image,label) in enumerate(train_dataset.take(16)):
  plt.subplot(4,4,i+1)
  plt.imshow(image)
  plt.title(dataset_info.features['label'].int2str(label))
  plt.axis("off")

# Data preprocessing

In [ ]:
def resizing(image,label):
  return tf.image.resize(image,(224,224))/255.0,label

In [ ]:
train_dataset=train_dataset.map(resizing)
val_dataset=val_dataset.map(resizing)
test_dataset=test_dataset.map(resizing)

In [ ]:
train_dataset=train_dataset.shuffle(buffer_size=8,reshuffle_each_iteration=True).batch(32).prefetch(tf.data.AUTOTUNE)
val_dataset=val_dataset.shuffle(buffer_size=8,reshuffle_each_iteration=True).batch(32).prefetch(tf.data.AUTOTUNE)
test_dataset=test_dataset.shuffle(buffer_size=8,reshuffle_each_iteration=True).batch(32).prefetch(tf.data.AUTOTUNE)

# Model Buliding

In [ ]:
model = Sequential()
model.add(InputLayer([224,224,3]))
model.add(Conv2D(filters=6, activation="relu", kernel_size=(3, 3), strides=1))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2), strides=2))
model.add(Conv2D(filters=6, activation="relu", kernel_size=(3, 3), strides=1))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2), strides=2))
model.add(Flatten())
model.add(Dense(19176, activation="relu"))
model.add(BatchNormalization())
model.add(Dense(100, activation="relu"))
model.add(BatchNormalization())
model.add(Dense(1, activation="sigmoid"))

In [ ]:
model.compile(optimizer='adam', 
                loss='binary_crossentropy', 
                metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
history=model.fit(train_dataset,validation_data=val_dataset,epochs=100,verbose=1)

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title("Model loss")
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train_loss','val_loss'])
plt.plot()

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title("Model acc")
plt.ylabel('acc')
plt.xlabel('epoch')
plt.legend(['train_acc','val_loss'])
plt.plot()

In [ ]:
model.evaluate(test_dataset)


In [ ]:
x=model.predict(test_dataset.take(1))[0][0]
def parasite_or_not(x):
    if(x>0.5):
        return str('U') 
    else:
        return str('P')


## Function API

In [ ]:
feature_input=Input(shape=(224,244,3),name="Input_Image")
x=Conv2D(filters=6,activation="relu",kernel_size=(3,3),strides=1)(feature_input)
x=BatchNormalization()(x)
x=MaxPooling2D(pool_size=(2,2),strides=2)(x)
x=Conv2D(filters=6,activation="relu",kernel_size=(3,3),strides=1)(x)
x=BatchNormalization()(x)
x=MaxPooling2D(pool_size=(2,2),strides=2)(x)
x=Flatten()(x)
x=Dense(19176,activation="relu")(x)
x=BatchNormalization()(x)
x=Dense(100,activation="relu")(x)
x=BatchNormalization()(x)
output=Dense(1,activation="sigmoid")(x)
lenet_model=Model(feature_input,output,name="Lenet_Model")


In [ ]:
lenet_model.summary()

In [ ]:
feature_input=Input(shape=(224,244,3),name="Input_Image")
x=Conv2D(filters=6,activation="relu",kernel_size=(3,3),strides=1)(feature_input)
x=BatchNormalization()(x)
x=MaxPooling2D(pool_size=(2,2),strides=2)(x)
x=Conv2D(filters=6,activation="relu",kernel_size=(3,3),strides=1)(x)
x=BatchNormalization()(x)
output=MaxPooling2D(pool_size=(2,2),strides=2)(x)
feature_extractor_model=Model(feature_input,output)


In [ ]:
feature_input=Input(shape=(224,244,3),name="Input_Image")
x=feature_extractor_model(feature_input)
x=Flatten()(x)
x=Dense(19176,activation="relu")(x)
x=BatchNormalization()(x)
x=Dense(100,activation="relu")(x)
x=BatchNormalization()(x)
output=Dense(1,activation="sigmoid")(x)
new_model=Model(feature_input,output)

In [ ]:
new_model.summary()

## Model Subclassing

In [ ]:
class FeatureExtractor(Layer):
    def __init__(self, filters, kernel_size, strides, padding, activation, pool_size):
        super(FeatureExtractor, self).__init__()
        self.conv_1 = Conv2D(filters=filters, kernel_size=kernel_size, strides=strides, 
                            padding=padding, activation=activation)
        self.batch_1 = BatchNormalization()
        self.pool_1 = MaxPooling2D(pool_size=pool_size, strides=strides)
        self.conv_2 = Conv2D(filters=filters*2, kernel_size=kernel_size, strides=strides, 
                            padding=padding, activation=activation)
        self.batch_2 = BatchNormalization()
        self.pool_2 = MaxPooling2D(pool_size=pool_size, strides=strides)

    def call(self, x, training=False):
        x = self.conv_1(x)
        x = self.batch_1(x, training=training)
        x = self.pool_1(x)
        x = self.conv_2(x)
        x = self.batch_2(x, training=training)
        x = self.pool_2(x)
        return x

feature_subclassed = FeatureExtractor(8, 3, 1, "valid", "relu", 2)

class LenetModel(Model):
    def __init__(self, filters=8, kernel_size=3, strides=1, padding="valid", activation="relu", pool_size=2):
        super(LenetModel, self).__init__()
        self.feature_extractor = FeatureExtractor(filters, kernel_size, strides, padding, activation, pool_size)
        self.flatten = Flatten()
        self.dense_1 = Dense(100, activation="relu")
        self.batch_1 = BatchNormalization()
        self.dense_2 = Dense(10, activation="relu")
        self.batch_2 = BatchNormalization()
        self.dense_3 = Dense(1, activation="sigmoid")

    def call(self, x, training=False):
        x = self.feature_extractor(x, training=training)
        x = self.flatten(x)
        x = self.dense_1(x)
        x = self.batch_1(x, training=training)
        x = self.dense_2(x)
        x = self.batch_2(x, training=training)
        x = self.dense_3(x)
        return x

lenet_sub_classed = LenetModel()

sample_input = tf.zeros((1, 224, 224, 3))
output = lenet_sub_classed(sample_input)

lenet_sub_classed.summary()


In [ ]:
class LossCallBack(Callback):
  def on_epoch_end(self,epoch,logs):
    print("\n For Epoch Number {} The Model has a loss of {}".fromat(epoch+1,logs["loss"]))
  def on_batch_end(self,batch,logs):
    print("\n For Batch Number {} The Model has a loss of {}".fromat(batch+1,logs))

In [ ]:
csvlogger=CSVLogger(
    'logs.csv', append=False
)
es_callback=EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=False,mode='auto',baseline=None,min_delta=0)

In [ ]:
def Scheduler(epoch,lr):
  if epoch<10:
    return lr
  else:
    return lr*tf.math.exp(-0.1)
scheduler_callback=LearningRateScheduler(Scheduler,verbose=0)

In [ ]:
metrics=[TruePositives(name="tp"),FalsePositives(name="fp"),TrueNegatives(name="tn"),FalseNegatives(name="fn"),BinaryAccuracy(name="accuracy"),Precision(name="precision"),Recall(name="recall"),AUC(name="auc")]

In [ ]:
lenet_sub_classed.compile(optimizer='Adam',loss='categorical_crossentropy',metrics=metrics)

## Data Augementation

In [ ]:
def resize_rescale(image,label):
    return tf.image.resize(image,(224,244))/255.0,label
def augment(image,label):
    image,label=resize_rescale(image,label)
    image=tf.image.rot90(image)
    image=tf.image.flip_left_right(image)
    return image,label
#train_dataset=train_dataset.map(augment)

train_dataset=(train_dataset.shuffle(buffer_size=8,reshuffle_each_iteration=True).map(augment).batch(32).prefetch(tf.data.AUTOTUNE))
val_dataset=(val_dataset.shuffle(buffer_size=8,reshuffle_each_iteration=True).map(resize_rescale).batch(32).prefetch(tf.data.AUTOTUNE))
test_dataset=(test_dataset.shuffle(buffer_size=8,reshuffle_each_iteration=True).map(resize_rescale).batch(32).prefetch(tf.data.AUTOTUNE))



## MixUp Augementation
### X=hX+(1-h)X
### label=hY+(1-h)Y

In [ ]:
import tensorflow_probability as tfds

train_dataset_1=train_dataset.shuffle(buffer_size=8,reshuffle_each_iteratior=True).map(resize_rescale)
train_dataset_2=train_dataset.shuffle(buffer_size=8,reshuffle_each_iteratior=True).map(resize_rescale)

mixed_dataset=tf.data.Dataset.zip((train_dataset_1,train_dataset_2))

def mixup(train_dataset_1,train_dataset_2):
    (image_1,label_1),(image_2,label_2)=train_dataset_1,train_dataset_2
    lamda=tfds.distributions.Beta(0.4,0.4)
    lamda=lamda.sample(1)[0]
    image=lamda*image_1 + (1-lamda)*image_2
    label=lamda*label_1 + (1-lamda)*label_2
    return image,label

train_dataset=(train_dataset.shuffle(buffer_size=8,reshuffle_each_iteration=True).map(mixup).batch(32).prefetch(tf.data.AUTOTUNE))
val_dataset=(val_dataset.shuffle(buffer_size=8,reshuffle_each_iteration=True).map(resize_rescale).batch(32).prefetch(tf.data.AUTOTUNE))
test_dataset=(test_dataset.shuffle(buffer_size=8,reshuffle_each_iteration=True).map(resize_rescale).batch(32).prefetch(tf.data.AUTOTUNE))

